<a href="https://colab.research.google.com/github/MajidSharaf/CampaignLens/blob/main/Pipeline/02_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: load cleaned data from step 1

In [1]:
!wget "https://raw.githubusercontent.com/MajidSharaf/CampaignLens/main/Datasets/Versions/comments_cleaned.csv"

--2026-06-12 21:06:39--  https://raw.githubusercontent.com/MajidSharaf/CampaignLens/main/Datasets/Versions/comments_cleaned.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2299025 (2.2M) [text/plain]
Saving to: ‘comments_cleaned.csv’

comments_cleaned.cs 100%[===================>]   2.19M  --.-KB/s    in 0.04s   

2026-06-12 21:06:39 (54.9 MB/s) - ‘comments_cleaned.csv’ saved [2299025/2299025]



In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
import spacy

!pip install spacy
!python -m spacy download en_core_web_sm

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 81.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [3]:
df = pd.read_csv("/content/comments_cleaned.csv")
print(len(df))
df.head()

22174


,cleaned_text
0,not_want news google
1,trump dumbest president american history terri...
2,fareed know full well reason many objected jcp...
3,china 's xinjiang genocide million visited xin...
4,jet pilot need precision


Part 2: NER with NLTK

In [4]:
# NLTK NER on first 5 comments
for i, comment in enumerate(df['cleaned_text'][:5]):
    print(f"Comment {i+1}: {comment}")
    tokens = nltk.word_tokenize(comment)
    tags = nltk.pos_tag(tokens)
    tree = nltk.ne_chunk(tags)
    print(tree)
    print()

Comment 1: not_want news google
(S not_want/JJ news/NN google/NN)

Comment 2: trump dumbest president american history terrible rollemodel america 's young people nothing violence stupidity lie
(S
  trump/NN
  dumbest/JJS
  president/NN
  american/JJ
  history/NN
  terrible/JJ
  rollemodel/NN
  america/NN
  's/POS
  young/JJ
  people/NNS
  nothing/NN
  violence/NN
  stupidity/NN
  lie/NN)

Comment 3: fareed know full well reason many objected jcpoa lifting sanction coupled sunset clause put iran world 's foremost sponsor terrorism legal funded path acquiring nuclear weapon pretend issue legitimized islamist regime disengenuous attempt lend credence embarrassment agreement outlived usefulness service no longer required
(S
  fareed/NN
  know/VBP
  full/JJ
  well/RB
  reason/NN
  many/JJ
  objected/VBD
  jcpoa/JJ
  lifting/VBG
  sanction/NN
  coupled/VBN
  sunset/NN
  clause/NN
  put/VBD
  iran/JJ
  world/NN
  's/POS
  foremost/NN
  sponsor/NN
  terrorism/NN
  legal/JJ
  funded/JJ
  path/

NLTK found almost no entities because preprocessing lowercased all text and NLTK relies on capitalization.

Part 3: NER with spaCy

In [5]:
nlp = spacy.load("en_core_web_sm")

for i, comment in enumerate(df['cleaned_text'][:5]):
    print(f"Comment {i+1}: {comment}")
    doc = nlp(comment)
    if doc.ents:
        for ent in doc.ents:
            print(f"  -> {ent.text} : {ent.label_}")
    else:
        print("  -> No entities found")
    print()

Comment 1: not_want news google
  -> No entities found

Comment 2: trump dumbest president american history terrible rollemodel america 's young people nothing violence stupidity lie
  -> trump dumbest : ORG
  -> american : NORP
  -> america : GPE

Comment 3: fareed know full well reason many objected jcpoa lifting sanction coupled sunset clause put iran world 's foremost sponsor terrorism legal funded path acquiring nuclear weapon pretend issue legitimized islamist regime disengenuous attempt lend credence embarrassment agreement outlived usefulness service no longer required
  -> iran : GPE
  -> islamist : NORP

Comment 4: china 's xinjiang genocide million visited xinjiang not_a single photo trace genocide lying farted zenophobia still blatantly smeared china thousand palesteninians dying hand izrale gennozide farted zealot not_blow wind
  -> china : GPE
  -> xinjiang : GPE
  -> million : CARDINAL
  -> xinjiang : GPE
  -> china : GPE
  -> thousand : CARDINAL

Comment 5: jet pilot ne

In [6]:
# run spaCy NER on all comments
results = []

for comment in df['cleaned_text']:
    doc = nlp(comment)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    results.append(entities)

NER_df = pd.DataFrame({'cleaned_text': df['cleaned_text'], 'entities': results})
print(NER_df.head())

                                        cleaned_text  \
0                               not_want news google   
1  trump dumbest president american history terri...   
2  fareed know full well reason many objected jcp...   
3  china 's xinjiang genocide million visited xin...   
4                           jet pilot need precision   

                                            entities  
0                                                 []  
1  [(trump dumbest, ORG), (american, NORP), (amer...  
2                    [(iran, GPE), (islamist, NORP)]  
3  [(china, GPE), (xinjiang, GPE), (million, CARD...  
4                                                 []  


In [7]:
print(len(NER_df))

22174


In [8]:
# most common entities
from collections import Counter
all_entities = [(ent, label) for entities in results for ent, label in entities]
entity_counts = Counter(all_entities)

top_entities = pd.DataFrame(entity_counts.most_common(20), columns=['Entity', 'Count'])
print(top_entities)

                                               Entity  Count
0                                        (china, GPE)   1940
1                                    (american, NORP)   1027
2                                     (one, CARDINAL)    756
3                       (face_with_tears_of_joy, ORG)    726
4                                      (america, GPE)    707
5                                     (chinese, NORP)    546
6                                    (democrat, NORP)    484
7                                         (iran, GPE)    408
8                                  (white house, ORG)    396
9                                    (first, ORDINAL)    362
10                                       (japan, GPE)    351
11                                (thumbs_up, PERSON)    248
12                                          (xi, ORG)    242
13  (face_with_tears_of_joy face_with_tears_of_joy...    236
14                                    (two, CARDINAL)    213
15                      

In [9]:
# save NER results
NER_df.to_csv('NER_results.csv', index=False)
print("NER results saved")

NER results saved
